In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from io import BytesIO
import os
import sys
import time
import json
import warnings

import numpy as np
import pandas as pd
import requests
import subprocess

try:
    import pyarrow  # noqa: F401
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyarrow"])
    import pyarrow  # noqa: F401

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)

warnings.filterwarnings("ignore", category=FutureWarning)

PROJECT_TITLE = "Against All Odds"
WEEK_NUMBER = 3
RANDOM_SEED = 42

np.random.seed(RANDOM_SEED)

SEASONS = {
    2021: {"label": "2021-22", "end_year": 2022, "split": "TRAIN"},
    2022: {"label": "2022-23", "end_year": 2023, "split": "TRAIN"},
    2023: {"label": "2023-24", "end_year": 2024, "split": "TRAIN"},
    2024: {"label": "2024-25", "end_year": 2025, "split": "VALIDATION"},
    2025: {"label": "2025-26", "end_year": 2026, "split": "TEST"},
}

RAW_BASE_URL = "https://raw.githubusercontent.com/llimllib/nba_data/main/data"
REQUEST_TIMEOUT = 90
MAX_RETRIES = 3

print(f"✅ {PROJECT_TITLE} — Week {WEEK_NUMBER}")
print("Selected NBA seasons:", [v["label"] for v in SEASONS.values()])

# ============================================================
# 2. Project directory setup
# ============================================================

def running_in_colab():
    return "google.colab" in sys.modules

CURRENT_DIR = Path.cwd().resolve()

KNOWN_REPO_NAMES = {
    "against-all-odds",
    "againstallodds",
    "Against-All-Odds",
    "Against_All_Odds",
    "Against All Odds",
}

if CURRENT_DIR.name in KNOWN_REPO_NAMES:
    PROJECT_ROOT = CURRENT_DIR
elif running_in_colab():
    # Matches the repository spelling used in the Week 2 notebook.
    PROJECT_ROOT = Path("/content/againstallodds")
else:
    PROJECT_ROOT = CURRENT_DIR / "againstallodds"

RAW_DIR = PROJECT_ROOT / "data" / "raw" / "nba"
CLEAN_DIR = PROJECT_ROOT / "data" / "clean" / "nba"
DOCS_DIR = PROJECT_ROOT / "docs"
RESULTS_DIR = PROJECT_ROOT / "results"

for folder in [RAW_DIR, CLEAN_DIR, DOCS_DIR, RESULTS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print(f"✅ Project root: {PROJECT_ROOT}")
print(f"✅ Raw NBA directory: {RAW_DIR}")
print(f"✅ Clean NBA directory: {CLEAN_DIR}")

# ============================================================
# 3. Canonical NBA team names
# ============================================================

NBA_TEAM_NAMES = {
    "ATL": "Atlanta Hawks",
    "BOS": "Boston Celtics",
    "BKN": "Brooklyn Nets",
    "CHA": "Charlotte Hornets",
    "CHI": "Chicago Bulls",
    "CLE": "Cleveland Cavaliers",
    "DAL": "Dallas Mavericks",
    "DEN": "Denver Nuggets",
    "DET": "Detroit Pistons",
    "GSW": "Golden State Warriors",
    "HOU": "Houston Rockets",
    "IND": "Indiana Pacers",
    "LAC": "Los Angeles Clippers",
    "LAL": "Los Angeles Lakers",
    "MEM": "Memphis Grizzlies",
    "MIA": "Miami Heat",
    "MIL": "Milwaukee Bucks",
    "MIN": "Minnesota Timberwolves",
    "NOP": "New Orleans Pelicans",
    "NYK": "New York Knicks",
    "OKC": "Oklahoma City Thunder",
    "ORL": "Orlando Magic",
    "PHI": "Philadelphia 76ers",
    "PHX": "Phoenix Suns",
    "POR": "Portland Trail Blazers",
    "SAC": "Sacramento Kings",
    "SAS": "San Antonio Spurs",
    "TOR": "Toronto Raptors",
    "UTA": "Utah Jazz",
    "WAS": "Washington Wizards",
}

NBA_TEAM_ABBRS = set(NBA_TEAM_NAMES)

assert len(NBA_TEAM_NAMES) == 30
print("✅ Canonical map contains 30 NBA teams")

# ============================================================
# 4. Download/cache helper
# ============================================================

SESSION = requests.Session()
SESSION.headers.update({
    "User-Agent": "Against-All-Odds-RIT-Capstone/1.0"
})


def download_parquet(url, destination, max_retries=MAX_RETRIES):
    destination = Path(destination)

    if destination.exists():
        print(f"✅ Using cached file: {destination.name}")
        return destination

    last_error = None

    for attempt in range(1, max_retries + 1):
        try:
            print(f"Downloading {url} (attempt {attempt}/{max_retries})")
            response = SESSION.get(url, timeout=REQUEST_TIMEOUT)
            response.raise_for_status()

            destination.write_bytes(response.content)
            print(f"✅ Saved raw file: {destination}")
            return destination

        except requests.RequestException as exc:
            last_error = exc
            if attempt < max_retries:
                time.sleep(2 * attempt)

    raise RuntimeError(
        f"Could not download {url}. Last error: {last_error}"
    )


def normalize_columns(df):
    out = df.copy()
    out.columns = [str(c).strip().lower() for c in out.columns]
    return out

# ============================================================
# 5. Collect all five historical NBA seasons
# ============================================================

raw_frames = []

for season_start, info in SEASONS.items():
    season_end = info["end_year"]
    season_label = info["label"]

    filename = f"gamelog_{season_end}.parquet"
    url = f"{RAW_BASE_URL}/{filename}"
    local_path = RAW_DIR / filename

    download_parquet(url, local_path)

    season_df = pd.read_parquet(local_path)
    season_df = normalize_columns(season_df)

    season_df["season_start"] = season_start
    season_df["season_end"] = season_end
    season_df["season_label"] = season_label
    season_df["data_split"] = info["split"]
    season_df["source_file"] = filename

    raw_frames.append(season_df)

    print(
        f"✅ {season_label}: "
        f"{len(season_df):,} raw team-game rows, "
        f"{len(season_df.columns)} columns"
    )

nba_raw = pd.concat(raw_frames, ignore_index=True)

print("\nCombined raw shape:", nba_raw.shape)
display(nba_raw.head())

# ============================================================
# 6. Validate expected raw schema
# ============================================================

REQUIRED_RAW_COLUMNS = [
    "team_id",
    "team_abbreviation",
    "team_name",
    "game_id",
    "game_date",
    "matchup",
    "wl",
    "pts",
]

missing_required_columns = [
    col for col in REQUIRED_RAW_COLUMNS
    if col not in nba_raw.columns
]

if missing_required_columns:
    raise ValueError(
        "Required NBA source columns are missing: "
        f"{missing_required_columns}"
    )

print("✅ Required raw NBA columns are present")

print("\nAvailable source columns:")
print(sorted(nba_raw.columns.tolist()))

# ============================================================
# 7. Keep regular-season games only and normalize identifiers
# ============================================================

nba = nba_raw.copy()

# Ensure stable string game IDs such as 0022100001.
nba["game_id"] = (
    nba["game_id"]
    .astype(str)
    .str.replace(r"\.0$", "", regex=True)
    .str.strip()
    .str.zfill(10)
)

before_regular_filter = len(nba)

# 002 = NBA regular season.
nba = nba[nba["game_id"].str.startswith("002")].copy()

print(
    f"Regular-season filter: "
    f"{before_regular_filter:,} -> {len(nba):,} team-game rows"
)

# Keep only the 30 NBA team abbreviations expected in this project.
nba["team_abbreviation"] = (
    nba["team_abbreviation"]
    .astype(str)
    .str.strip()
    .str.upper()
)

unexpected_abbrs = sorted(
    set(nba["team_abbreviation"].dropna()) - NBA_TEAM_ABBRS
)

if unexpected_abbrs:
    print("⚠️ Unexpected team abbreviations:", unexpected_abbrs)

nba = nba[nba["team_abbreviation"].isin(NBA_TEAM_ABBRS)].copy()

print("✅ Restricted data to the 30 project NBA teams")

# ============================================================
# 8. Standardize team names, dates, outcome text, and home/away
# ============================================================

# Canonical team names based on abbreviation.
nba["team_name_source"] = nba["team_name"]
nba["team_name"] = nba["team_abbreviation"].map(NBA_TEAM_NAMES)

# Parse date consistently.
nba["game_date"] = pd.to_datetime(
    nba["game_date"],
    errors="coerce"
)

# Standardize W/L text.
nba["wl"] = (
    nba["wl"]
    .astype(str)
    .str.strip()
    .str.upper()
    .replace({"NAN": np.nan, "NONE": np.nan})
)

# Home/away identification from NBA matchup notation.
# Example home: BOS vs. NYK
# Example away: NYK @ BOS
matchup_text = nba["matchup"].astype(str)

nba["is_home"] = np.where(
    matchup_text.str.contains(r"\bvs\.", regex=True, na=False),
    1,
    np.where(
        matchup_text.str.contains("@", regex=False, na=False),
        0,
        np.nan,
    )
)

nba["location"] = np.where(
    nba["is_home"].eq(1),
    "HOME",
    np.where(nba["is_home"].eq(0), "AWAY", pd.NA),
)

print("Home/away counts:")
display(nba["location"].value_counts(dropna=False).rename_axis("location").reset_index(name="rows"))

# ============================================================
# 9. Remove duplicates and audit critical missing values
# ============================================================

rows_before_dedup = len(nba)

duplicate_mask = nba.duplicated(
    subset=["game_id", "team_id"],
    keep="first"
)

duplicate_team_game_rows = int(duplicate_mask.sum())

nba = (
    nba.loc[~duplicate_mask]
    .copy()
    .reset_index(drop=True)
)

print(f"Duplicate team-game rows removed: {duplicate_team_game_rows}")
print(f"Rows after de-duplication: {len(nba):,}")

CRITICAL_COLUMNS = [
    "game_id",
    "game_date",
    "team_id",
    "team_abbreviation",
    "team_name",
    "matchup",
    "wl",
    "pts",
    "is_home",
]

critical_missing_before = nba[CRITICAL_COLUMNS].isna().sum()

print("\nCritical missing values before removal:")
display(
    critical_missing_before
    .rename("missing_count")
    .reset_index()
    .rename(columns={"index": "column"})
)

critical_missing_row_mask = nba[CRITICAL_COLUMNS].isna().any(axis=1)
rows_with_critical_missing = int(critical_missing_row_mask.sum())

if rows_with_critical_missing:
    print(
        f"⚠️ Dropping {rows_with_critical_missing} rows "
        "with critical missing values."
    )
    nba = nba.loc[~critical_missing_row_mask].copy()

print(f"✅ Critical-data cleaning complete: {len(nba):,} rows remain")

# ============================================================
# 10. Full missing-value audit
# ============================================================

missingness = pd.DataFrame({
    "column": nba.columns,
    "missing_count": nba.isna().sum().values,
    "missing_pct": (nba.isna().mean().values * 100).round(3),
    "dtype": [str(dtype) for dtype in nba.dtypes],
}).sort_values(
    ["missing_pct", "column"],
    ascending=[False, True]
).reset_index(drop=True)

display(missingness.head(40))

missingness_path = DOCS_DIR / "week3_nba_missingness.csv"
missingness.to_csv(missingness_path, index=False)

print(f"✅ Missing-value audit saved: {missingness_path}")

# ============================================================
# 11. Verify each game has exactly one home and one away row
# ============================================================

game_structure = (
    nba.groupby("game_id")
    .agg(
        rows=("team_id", "size"),
        unique_teams=("team_id", "nunique"),
        home_rows=("is_home", lambda s: int((s == 1).sum())),
        away_rows=("is_home", lambda s: int((s == 0).sum())),
    )
    .reset_index()
)

valid_game_mask = (
    game_structure["rows"].eq(2)
    & game_structure["unique_teams"].eq(2)
    & game_structure["home_rows"].eq(1)
    & game_structure["away_rows"].eq(1)
)

invalid_games = game_structure.loc[~valid_game_mask].copy()

print(f"Total candidate games: {len(game_structure):,}")
print(f"Structurally valid games: {valid_game_mask.sum():,}")
print(f"Invalid games: {len(invalid_games):,}")

if not invalid_games.empty:
    display(invalid_games.head(20))

valid_game_ids = set(
    game_structure.loc[valid_game_mask, "game_id"]
)

nba = nba[nba["game_id"].isin(valid_game_ids)].copy()

print("✅ Retained only games with one home and one away team row")

# ============================================================
# 12. Add derived opponent points and win indicator to team logs
# ============================================================

nba["pts"] = pd.to_numeric(nba["pts"], errors="coerce")

if "plus_minus" in nba.columns:
    nba["plus_minus"] = pd.to_numeric(
        nba["plus_minus"],
        errors="coerce"
    )
    nba["opp_pts"] = nba["pts"] - nba["plus_minus"]
else:
    nba["opp_pts"] = np.nan

nba["team_win"] = nba["wl"].eq("W").astype(int)

# Chronological order for all future feature engineering.
nba = nba.sort_values(
    ["game_date", "game_id", "is_home"],
    ascending=[True, True, False]
).reset_index(drop=True)

print("✅ Team-game rows sorted chronologically")
display(
    nba[
        [
            "season_label",
            "game_date",
            "game_id",
            "team_abbreviation",
            "location",
            "wl",
            "pts",
            "opp_pts",
        ]
    ].head(10)
)

# ============================================================
# 13. Build one clean game-level row per NBA game
# ============================================================

TEAM_STATS_TO_KEEP = [
    "pts",
    "fgm",
    "fga",
    "fg_pct",
    "fg3m",
    "fg3a",
    "fg3_pct",
    "ftm",
    "fta",
    "ft_pct",
    "oreb",
    "dreb",
    "reb",
    "ast",
    "stl",
    "blk",
    "tov",
    "pf",
    "plus_minus",
    "off_rating",
    "def_rating",
    "net_rating",
    "efg_pct",
    "ts_pct",
    "pace",
    "poss",
]

TEAM_STATS_TO_KEEP = [
    col for col in TEAM_STATS_TO_KEEP
    if col in nba.columns
]

base_identity = [
    "game_id",
    "season_start",
    "season_end",
    "season_label",
    "data_split",
    "game_date",
]

home = nba[nba["is_home"].eq(1)].copy()
away = nba[nba["is_home"].eq(0)].copy()

home_keep = base_identity + [
    "team_id",
    "team_abbreviation",
    "team_name",
    "wl",
] + TEAM_STATS_TO_KEEP

away_keep = ["game_id"] + [
    "team_id",
    "team_abbreviation",
    "team_name",
    "wl",
] + TEAM_STATS_TO_KEEP

home = home[home_keep].copy()
away = away[away_keep].copy()

home = home.rename(columns={
    "team_id": "home_team_id",
    "team_abbreviation": "home_team_abbr",
    "team_name": "home_team_name",
    "wl": "home_wl",
    **{c: f"home_{c}" for c in TEAM_STATS_TO_KEEP},
})

away = away.rename(columns={
    "team_id": "away_team_id",
    "team_abbreviation": "away_team_abbr",
    "team_name": "away_team_name",
    "wl": "away_wl",
    **{c: f"away_{c}" for c in TEAM_STATS_TO_KEEP},
})

nba_games = home.merge(
    away,
    on="game_id",
    how="inner",
    validate="one_to_one",
)

nba_games["home_is_home"] = 1
nba_games["away_is_home"] = 0

# Use the official W/L fields as the primary outcome source.
valid_home_win = nba_games["home_wl"].eq("W") & nba_games["away_wl"].eq("L")
valid_away_win = nba_games["home_wl"].eq("L") & nba_games["away_wl"].eq("W")

nba_games["HOME_WIN"] = np.select(
    [valid_home_win, valid_away_win],
    [1, 0],
    default=np.nan,
)

invalid_wl_mask = nba_games["HOME_WIN"].isna()
if invalid_wl_mask.any():
    print(
        f"⚠️ Found {int(invalid_wl_mask.sum())} game(s) with inconsistent W/L fields."
    )
    display(
        nba_games.loc[
            invalid_wl_mask,
            [
                "game_id", "game_date",
                "home_team_abbr", "away_team_abbr",
                "home_wl", "away_wl", "home_pts", "away_pts",
            ],
        ]
    )

    # Fall back to score only when the W/L pair itself is unusable.
    score_not_tied = nba_games["home_pts"].ne(nba_games["away_pts"])
    fallback_mask = invalid_wl_mask & score_not_tied
    nba_games.loc[fallback_mask, "HOME_WIN"] = (
        nba_games.loc[fallback_mask, "home_pts"]
        > nba_games.loc[fallback_mask, "away_pts"]
    ).astype(int)

if nba_games["HOME_WIN"].isna().any():
    raise ValueError(
        "Some games have neither a valid W/L pair nor a usable final score."
    )

nba_games["HOME_WIN"] = nba_games["HOME_WIN"].astype(int)

# Initial score-based margin and total. These may be repaired below if the
# source contains a tied score for a game whose W/L fields identify a winner.
nba_games["HOME_MARGIN"] = nba_games["home_pts"] - nba_games["away_pts"]
nba_games["GAME_TOTAL_POINTS"] = nba_games["home_pts"] + nba_games["away_pts"]
nba_games["score_quality"] = "OK"

# NBA games cannot officially end tied. If the source points are tied, first
# try to reconstruct the score using each team's plus/minus field.
tie_mask = nba_games["home_pts"].eq(nba_games["away_pts"])
tie_count = int(tie_mask.sum())

if tie_count:
    print(f"⚠️ Found {tie_count} tied-score record(s) in the source data.")
    display(
        nba_games.loc[
            tie_mask,
            [
                "game_id", "season_label", "game_date",
                "home_team_abbr", "away_team_abbr",
                "home_pts", "away_pts", "home_wl", "away_wl",
            ],
        ]
    )

    repaired_mask = pd.Series(False, index=nba_games.index)

    if {"home_plus_minus", "away_plus_minus"}.issubset(nba_games.columns):
        home_pm = pd.to_numeric(nba_games["home_plus_minus"], errors="coerce")
        away_pm = pd.to_numeric(nba_games["away_plus_minus"], errors="coerce")

        # Reconstruct each opponent's points independently.
        repaired_home_pts = nba_games["away_pts"] - away_pm
        repaired_away_pts = nba_games["home_pts"] - home_pm

        repairable = (
            tie_mask
            & home_pm.notna()
            & away_pm.notna()
            & np.isclose(home_pm, -away_pm)
            & home_pm.ne(0)
            & repaired_home_pts.notna()
            & repaired_away_pts.notna()
        )

        # The reconstructed margin must agree with the official W/L result.
        reconstructed_margin = repaired_home_pts - repaired_away_pts
        outcome_agrees = (
            (nba_games["HOME_WIN"].eq(1) & reconstructed_margin.gt(0))
            | (nba_games["HOME_WIN"].eq(0) & reconstructed_margin.lt(0))
        )
        repairable &= outcome_agrees

        if repairable.any():
            nba_games.loc[repairable, "home_pts"] = repaired_home_pts.loc[repairable]
            nba_games.loc[repairable, "away_pts"] = repaired_away_pts.loc[repairable]
            nba_games.loc[repairable, "HOME_MARGIN"] = reconstructed_margin.loc[repairable]
            nba_games.loc[repairable, "GAME_TOTAL_POINTS"] = (
                repaired_home_pts.loc[repairable] + repaired_away_pts.loc[repairable]
            )
            nba_games.loc[repairable, "score_quality"] = "REPAIRED_FROM_PLUS_MINUS"
            repaired_mask = repairable
            print(f"✅ Repaired {int(repairable.sum())} tied-score record(s) using plus/minus.")

    unresolved_ties = tie_mask & ~repaired_mask

    if unresolved_ties.any():
        # Keep the official W/L outcome, but do not invent a scoring margin.
        # Mark the score fields as unresolved so downstream analysis can exclude
        # these rows from score-based calculations without losing the game outcome.
        nba_games.loc[unresolved_ties, "HOME_MARGIN"] = np.nan
        nba_games.loc[unresolved_ties, "GAME_TOTAL_POINTS"] = np.nan
        nba_games.loc[unresolved_ties, "score_quality"] = "UNRESOLVED_SOURCE_SCORE"

        print(
            f"⚠️ {int(unresolved_ties.sum())} tied-score record(s) could not be "
            "reconstructed. The official W/L result was retained, while score-based "
            "derived fields were set to missing."
        )

nba_games = (
    nba_games
    .sort_values(["game_date", "game_id"])
    .drop_duplicates(subset=["game_id"], keep="first")
    .reset_index(drop=True)
)

print(f"✅ Game-level dataset created: {len(nba_games):,} games")
display(nba_games.head())

# ============================================================
# 14. Validate historical coverage
# ============================================================

season_rows = []

for season_label, part in nba_games.groupby("season_label", sort=True):
    teams = set(part["home_team_abbr"]).union(set(part["away_team_abbr"]))

    season_rows.append({
        "season_start": int(part["season_start"].iloc[0]),
        "season_label": season_label,
        "data_split": part["data_split"].iloc[0],
        "games": int(part["game_id"].nunique()),
        "unique_teams": len(teams),
        "first_game": part["game_date"].min(),
        "last_game": part["game_date"].max(),
        "expected_regular_season_games": 1230,
    })

season_coverage = pd.DataFrame(season_rows)

season_coverage["game_count_pass"] = (
    season_coverage["games"]
    == season_coverage["expected_regular_season_games"]
)
season_coverage["team_count_pass"] = (
    season_coverage["unique_teams"] == 30
)

display(season_coverage)

coverage_path = RESULTS_DIR / "week3_nba_season_coverage.csv"
season_coverage.to_csv(coverage_path, index=False)

print(f"✅ Coverage report saved: {coverage_path}")

# ============================================================
# 15. Create team-season statistics
# ============================================================

TEAM_NUMERIC_STATS = [
    "pts",
    "opp_pts",
    "plus_minus",
    "fg_pct",
    "fg3_pct",
    "ft_pct",
    "reb",
    "ast",
    "stl",
    "blk",
    "tov",
    "off_rating",
    "def_rating",
    "net_rating",
    "efg_pct",
    "ts_pct",
    "pace",
]

TEAM_NUMERIC_STATS = [
    col for col in TEAM_NUMERIC_STATS
    if col in nba.columns
]

agg_dict = {
    "games": ("game_id", "nunique"),
    "wins": ("team_win", "sum"),
    "avg_points": ("pts", "mean"),
}

if "opp_pts" in nba.columns:
    agg_dict["avg_points_allowed"] = ("opp_pts", "mean")

if "plus_minus" in nba.columns:
    agg_dict["avg_point_differential"] = ("plus_minus", "mean")

for col in TEAM_NUMERIC_STATS:
    if col in {"pts", "opp_pts", "plus_minus"}:
        continue
    agg_dict[f"avg_{col}"] = (col, "mean")

team_season_stats = (
    nba.groupby(
        [
            "season_start",
            "season_label",
            "data_split",
            "team_id",
            "team_abbreviation",
            "team_name",
        ],
        as_index=False,
    )
    .agg(**agg_dict)
)

team_season_stats["losses"] = (
    team_season_stats["games"]
    - team_season_stats["wins"]
)

team_season_stats["win_pct"] = (
    team_season_stats["wins"]
    / team_season_stats["games"]
)

# Reorder key fields.
front_cols = [
    "season_start",
    "season_label",
    "data_split",
    "team_id",
    "team_abbreviation",
    "team_name",
    "games",
    "wins",
    "losses",
    "win_pct",
]

remaining_cols = [
    c for c in team_season_stats.columns
    if c not in front_cols
]

team_season_stats = team_season_stats[
    front_cols + remaining_cols
].sort_values(
    ["season_start", "win_pct"],
    ascending=[True, False]
).reset_index(drop=True)

print(f"✅ Team-season summary created: {len(team_season_stats):,} rows")
display(team_season_stats.head(15))

# ============================================================
# 16. Final quality-control audit
# ============================================================

game_required = [
    "game_id",
    "season_start",
    "season_label",
    "data_split",
    "game_date",
    "home_team_id",
    "home_team_abbr",
    "home_team_name",
    "away_team_id",
    "away_team_abbr",
    "away_team_name",
    "home_pts",
    "away_pts",
    "home_is_home",
    "away_is_home",
    "HOME_WIN",
]

quality_checks = {
    "five_selected_seasons": nba_games["season_label"].nunique() == 5,
    "game_id_unique": nba_games["game_id"].is_unique,
    "no_required_game_missing": not nba_games[game_required].isna().any().any(),
    "home_win_binary": set(nba_games["HOME_WIN"].unique()).issubset({0, 1}),
    "no_official_ties_after_repair": not (
        nba_games["home_pts"].eq(nba_games["away_pts"])
        & nba_games["score_quality"].ne("UNRESOLVED_SOURCE_SCORE")
    ).any(),
    "outcome_matches_wl": (
        ((nba_games["HOME_WIN"] == 1) & nba_games["home_wl"].eq("W") & nba_games["away_wl"].eq("L"))
        | ((nba_games["HOME_WIN"] == 0) & nba_games["home_wl"].eq("L") & nba_games["away_wl"].eq("W"))
    ).all(),
    "home_indicator_correct": nba_games["home_is_home"].eq(1).all(),
    "away_indicator_correct": nba_games["away_is_home"].eq(0).all(),
    "different_teams_each_game": (
        nba_games["home_team_id"] != nba_games["away_team_id"]
    ).all(),
    "chronologically_sorted": nba_games["game_date"].is_monotonic_increasing,
    "30_teams_each_season": season_coverage["team_count_pass"].all(),
    "1230_games_each_season": season_coverage["game_count_pass"].all(),
}

quality_report = pd.DataFrame([
    {"check": check, "passed": bool(passed)}
    for check, passed in quality_checks.items()
])

display(quality_report)

if not quality_report["passed"].all():
    print(
        "⚠️ One or more quality checks did not pass. "
        "Review the report before moving to modeling."
    )
else:
    print("✅ All Week 3 NBA quality checks passed")

# ============================================================
# 17. Save clean NBA datasets
# ============================================================

team_log_path = CLEAN_DIR / "nba_team_game_logs_clean_2021_22_to_2025_26.csv"
game_path = CLEAN_DIR / "nba_games_clean_2021_22_to_2025_26.csv"
team_summary_path = CLEAN_DIR / "nba_team_season_summary_2021_22_to_2025_26.csv"
quality_path = RESULTS_DIR / "week3_nba_cleaning_quality_report.csv"

nba.to_csv(team_log_path, index=False)
nba_games.to_csv(game_path, index=False)
team_season_stats.to_csv(team_summary_path, index=False)
quality_report.to_csv(quality_path, index=False)

print("✅ Saved clean team-game logs:")
print(team_log_path)

print("\n✅ Saved clean one-row-per-game dataset:")
print(game_path)

print("\n✅ Saved team-season summary:")
print(team_summary_path)

print("\n✅ Saved cleaning quality report:")
print(quality_path)

# ============================================================
# 18. Update the Week 1 data dictionary
# ============================================================

DATA_DICTIONARY_COLUMNS = [
    "variable_name",
    "sport",
    "level",
    "category",
    "data_type",
    "description",
    "source",
    "pre_game_only",
    "calculation_or_definition",
    "missing_value_rule",
    "required_or_optional",
    "notes",
]

template_path = DOCS_DIR / "data_dictionary_template.csv"

if template_path.exists():
    data_dictionary = pd.read_csv(template_path)
else:
    # Recreate the Week 1 baseline if the notebook is run in a fresh Colab session.
    data_dictionary = pd.DataFrame([
        {
            "variable_name": "game_id",
            "sport": "ALL",
            "level": "game",
            "category": "identifier",
            "data_type": "string",
            "description": "Unique game or match identifier",
            "source": "TBD",
            "pre_game_only": "yes",
            "calculation_or_definition": "Source ID or reproducible composite key",
            "missing_value_rule": "Must not be missing",
            "required_or_optional": "required",
            "notes": "Primary key",
        },
        {
            "variable_name": "game_date",
            "sport": "ALL",
            "level": "game",
            "category": "time",
            "data_type": "date",
            "description": "Date of the game or match",
            "source": "TBD",
            "pre_game_only": "yes",
            "calculation_or_definition": "Convert to consistent datetime format",
            "missing_value_rule": "Must not be missing",
            "required_or_optional": "required",
            "notes": "Used for chronological validation",
        },
        {
            "variable_name": "elo_diff",
            "sport": "ALL",
            "level": "game",
            "category": "team_strength",
            "data_type": "float",
            "description": "Pre-game home Elo minus away Elo",
            "source": "Derived",
            "pre_game_only": "yes",
            "calculation_or_definition": "home_pre_game_elo - away_pre_game_elo",
            "missing_value_rule": "Document handling",
            "required_or_optional": "required",
            "notes": "Calculate before current-game Elo update",
        },
        {
            "variable_name": "fragility_score",
            "sport": "ALL",
            "level": "game",
            "category": "reliability",
            "data_type": "float",
            "description": "Final prediction fragility score",
            "source": "Derived",
            "pre_game_only": "yes",
            "calculation_or_definition": "Defined after perturbation experiments",
            "missing_value_rule": "Not available in early weeks",
            "required_or_optional": "required_final",
            "notes": "Do not invent final formula in Week 3",
        },
        {
            "variable_name": "reliability_label",
            "sport": "ALL",
            "level": "game",
            "category": "reliability",
            "data_type": "string",
            "description": "Predict, Caution, or Abstain",
            "source": "Derived",
            "pre_game_only": "yes",
            "calculation_or_definition": "Thresholds selected using validation data",
            "missing_value_rule": "Not available in early weeks",
            "required_or_optional": "required_final",
            "notes": "Finalized later in the project",
        },
    ], columns=DATA_DICTIONARY_COLUMNS)

week3_dictionary_rows = [
    ["season_start", "NBA", "game", "time", "integer",
     "Starting year of the NBA season", "Derived from selected season",
     "yes", "Example: 2025 represents 2025-26", "Must not be missing",
     "required", "Used for chronological split"],
    ["season_label", "NBA", "game", "time", "string",
     "Readable NBA season label", "Derived", "yes",
     "YYYY-YY format", "Must not be missing", "required",
     "Example: 2025-26"],
    ["data_split", "NBA", "game", "validation", "string",
     "Chronological TRAIN/VALIDATION/TEST label", "Derived", "yes",
     "2021-24 train; 2024-25 validation; 2025-26 test",
     "Must not be missing", "required", "Never randomly shuffle across seasons"],
    ["home_team_id", "NBA", "game", "identifier", "integer",
     "NBA team ID for home team", "NBA Stats game log", "yes",
     "Home row identified from matchup notation", "Must not be missing",
     "required", ""],
    ["home_team_abbr", "NBA", "game", "identifier", "string",
     "Canonical home team abbreviation", "NBA Stats game log", "yes",
     "Standardized 3-letter abbreviation", "Must not be missing",
     "required", ""],
    ["home_team_name", "NBA", "game", "identifier", "string",
     "Canonical home team name", "Derived from abbreviation map", "yes",
     "Mapped from team abbreviation", "Must not be missing",
     "required", ""],
    ["away_team_id", "NBA", "game", "identifier", "integer",
     "NBA team ID for away team", "NBA Stats game log", "yes",
     "Away row identified from matchup notation", "Must not be missing",
     "required", ""],
    ["away_team_abbr", "NBA", "game", "identifier", "string",
     "Canonical away team abbreviation", "NBA Stats game log", "yes",
     "Standardized 3-letter abbreviation", "Must not be missing",
     "required", ""],
    ["away_team_name", "NBA", "game", "identifier", "string",
     "Canonical away team name", "Derived from abbreviation map", "yes",
     "Mapped from team abbreviation", "Must not be missing",
     "required", ""],
    ["home_pts", "NBA", "game", "outcome", "numeric",
     "Final points scored by home team", "NBA Stats game log", "no",
     "Final game score", "Must not be missing", "required",
     "Post-game value; not a same-game predictor"],
    ["away_pts", "NBA", "game", "outcome", "numeric",
     "Final points scored by away team", "NBA Stats game log", "no",
     "Final game score", "Must not be missing", "required",
     "Post-game value; not a same-game predictor"],
    ["HOME_WIN", "NBA", "game", "outcome", "integer",
     "Binary target indicating a home-team win", "Derived", "no",
     "1 when home W/L is W/L; 0 when home W/L is L/W; score fallback only if needed",
     "Must not be missing", "required", "Primary binary target for NBA modeling"],
    ["HOME_MARGIN", "NBA", "game", "outcome", "numeric",
     "Home-team final scoring margin", "Derived", "no",
     "home_pts - away_pts after score repair when possible",
     "May be missing only for unresolved malformed source scores", "required",
     "Post-game outcome information"],
    ["is_home", "NBA", "team_game", "context", "integer",
     "Home/away indicator on team-game rows", "Derived from matchup",
     "yes", "1 for vs.; 0 for @", "Must not be missing",
     "required", "Used to construct one-row-per-game dataset"],
    ["fg_pct", "NBA", "team_game", "offense", "float",
     "Current-game field-goal percentage", "NBA Stats game log", "no",
     "Source team box-score statistic", "May be missing in unusual records",
     "preferred", "Can only enter later-game predictors after shifting"],
    ["fg3_pct", "NBA", "team_game", "offense", "float",
     "Current-game three-point percentage", "NBA Stats game log", "no",
     "Source team box-score statistic", "May be missing in unusual records",
     "preferred", "Can only enter later-game predictors after shifting"],
    ["off_rating", "NBA", "team_game", "offense", "float",
     "Current-game offensive rating", "NBA Stats game log", "no",
     "Source advanced team statistic", "Document missingness",
     "preferred", "Use only historically with leakage-safe shifting"],
    ["def_rating", "NBA", "team_game", "defense", "float",
     "Current-game defensive rating", "NBA Stats game log", "no",
     "Source advanced team statistic", "Document missingness",
     "preferred", "Use only historically with leakage-safe shifting"],
    ["net_rating", "NBA", "team_game", "team_strength", "float",
     "Current-game net rating", "NBA Stats game log", "no",
     "off_rating - def_rating or source equivalent", "Document missingness",
     "preferred", "Use only historically with leakage-safe shifting"],
    ["efg_pct", "NBA", "team_game", "offense", "float",
     "Current-game effective field-goal percentage", "NBA Stats game log",
     "no", "Source advanced team statistic", "Document missingness",
     "preferred", "Use only historically with leakage-safe shifting"],
    ["pace", "NBA", "team_game", "context", "float",
     "Current-game pace estimate", "NBA Stats game log", "no",
     "Source advanced team statistic", "Document missingness",
     "optional", "Use only historically with leakage-safe shifting"],
]

week3_dictionary_df = pd.DataFrame(
    week3_dictionary_rows,
    columns=DATA_DICTIONARY_COLUMNS
)

# Remove existing NBA entries with the same variable/level before appending.
keys_to_replace = set(
    zip(
        week3_dictionary_df["variable_name"],
        week3_dictionary_df["sport"],
        week3_dictionary_df["level"],
    )
)

keep_mask = [
    (row.variable_name, row.sport, row.level) not in keys_to_replace
    for row in data_dictionary.itertuples(index=False)
]

data_dictionary = pd.concat(
    [
        data_dictionary.loc[keep_mask, DATA_DICTIONARY_COLUMNS],
        week3_dictionary_df,
    ],
    ignore_index=True,
)

week3_dictionary_path = DOCS_DIR / "data_dictionary_week3.csv"
data_dictionary.to_csv(week3_dictionary_path, index=False)

print(f"✅ Updated data dictionary saved: {week3_dictionary_path}")
display(data_dictionary.tail(len(week3_dictionary_df)))

# ============================================================
# 19. Save Week 3 source/cleaning documentation
# ============================================================

source_note = f'''# Week 3 NBA Data Source and Cleaning Plan

Generated: {datetime.now(timezone.utc).isoformat(timespec="seconds")}

## Seasons
- 2021-22
- 2022-23
- 2023-24
- 2024-25
- 2025-26

## Source
Public NBA team game-log parquet files from:
https://github.com/llimllib/nba_data

The source repository documents the files as NBA team-level game logs sourced
from stats.nba.com.

## Season file mapping
- 2021-22 -> gamelog_2022.parquet
- 2022-23 -> gamelog_2023.parquet
- 2023-24 -> gamelog_2024.parquet
- 2024-25 -> gamelog_2025.parquet
- 2025-26 -> gamelog_2026.parquet

## Cleaning rules
1. Keep regular-season game IDs beginning with 002.
2. Keep the 30 canonical NBA franchises.
3. Standardize names from team abbreviations.
4. Convert game_date with pandas datetime parsing.
5. Drop duplicate team-game records by game_id + team_id.
6. Remove rows only when critical identifiers/outcomes are missing.
7. Require two team rows per game: one home and one away.
8. Build one game-level record per game.
9. Create HOME_WIN primarily from the official home/away W/L pair.
10. Repair tied source scores from plus/minus when the reconstruction agrees with W/L.
11. If a malformed tied score cannot be repaired, retain the official outcome and mark score-based derived fields missing.
12. Sort all records chronologically.

## Leakage rule
Current-game box-score and advanced statistics are post-game data.
They are retained as historical information only. Week 6 rolling predictors
must use shift(1) before rolling calculations.
'''

source_note_path = DOCS_DIR / "week3_nba_source_and_cleaning_plan.md"
source_note_path.write_text(source_note, encoding="utf-8")

print(f"✅ Documentation saved: {source_note_path}")

# ============================================================
# 20. Final Week 3 cleaning status
# ============================================================

files_expected = {
    "clean_team_game_logs": team_log_path.exists(),
    "clean_game_dataset": game_path.exists(),
    "team_season_summary": team_summary_path.exists(),
    "updated_data_dictionary": week3_dictionary_path.exists(),
    "missingness_report": missingness_path.exists(),
    "coverage_report": coverage_path.exists(),
    "quality_report": quality_path.exists(),
    "source_documentation": source_note_path.exists(),
}

file_status = pd.DataFrame([
    {"deliverable": key, "created": bool(value)}
    for key, value in files_expected.items()
])

display(file_status)

print("=" * 78)
print("AGAINST ALL ODDS — WEEK 3 NBA DATA CLEANING")
print("=" * 78)

if quality_report["passed"].all() and file_status["created"].all():
    print("✅ WEEK 3 NBA CLEANING DELIVERABLE COMPLETE")
else:
    print("⚠️ WEEK 3 COMPLETED WITH ITEMS TO REVIEW")

print(f"\nClean games: {len(nba_games):,}")
print(f"Clean team-game rows: {len(nba):,}")
print(f"Seasons: {nba_games['season_label'].nunique()}")
print(f"Date range: {nba_games['game_date'].min().date()} to {nba_games['game_date'].max().date()}")

print("\nNext: run 04_nba_analysis.ipynb")


✅ Against All Odds — Week 3
Selected NBA seasons: ['2021-22', '2022-23', '2023-24', '2024-25', '2025-26']
✅ Project root: /content/againstallodds
✅ Raw NBA directory: /content/againstallodds/data/raw/nba
✅ Clean NBA directory: /content/againstallodds/data/clean/nba
✅ Canonical map contains 30 NBA teams
✅ Saved raw file: /content/againstallodds/data/raw/nba/gamelog_2022.parquet
✅ 2021-22: 2,460 raw team-game rows, 97 columns
✅ Saved raw file: /content/againstallodds/data/raw/nba/gamelog_2023.parquet
✅ 2022-23: 2,460 raw team-game rows, 97 columns
✅ Saved raw file: /content/againstallodds/data/raw/nba/gamelog_2024.parquet
✅ 2023-24: 2,460 raw team-game rows, 97 columns
✅ Saved raw file: /content/againstallodds/data/raw/nba/gamelog_2025.parquet
✅ 2024-25: 2,460 raw team-game rows, 96 columns
✅ Saved raw file: /content/againstallodds/data/raw/nba/gamelog_2026.parquet
✅ 2025-26: 2,802 raw team-game rows, 96 columns

Combined raw shape: (12642, 97)


,game_n,season_year,team_id,team_abbreviation,team_name,game_id,game_date,matchup,wl,min,...,efg_pct_rank,ts_pct_rank,pace_rank,pie_rank,available_flag,season_start,season_end,season_label,data_split,source_file
0,30.0,2021-22,1610612755,PHI,Philadelphia 76ers,0022101214,2022-04-09T00:00:00,PHI vs. IND,W,48.0,...,242,216,399,543,NaN,2021,2022,2021-22,TRAIN,gamelog_2022.parquet
1,31.0,2021-22,1610612754,IND,Indiana Pacers,0022101214,2022-04-09T00:00:00,IND @ PHI,L,48.0,...,489,326,399,1918,NaN,2021,2022,2021-22,TRAIN,gamelog_2022.parquet
2,32.0,2021-22,1610612746,LAC,LA Clippers,0022101212,2022-04-09T00:00:00,LAC vs. SAC,W,48.0,...,321,387,1005,638,NaN,2021,2022,2021-22,TRAIN,gamelog_2022.parquet
3,33.0,2021-22,1610612744,GSW,Golden State Warriors,0022101215,2022-04-09T00:00:00,GSW @ SAS,W,48.0,...,2152,1840,1461,611,NaN,2021,2022,2021-22,TRAIN,gamelog_2022.parquet
4,34.0,2021-22,1610612740,NOP,New Orleans Pelicans,0022101213,2022-04-09T00:00:00,NOP @ MEM,L,48.0,...,1869,1694,109,2387,NaN,2021,2022,2021-22,TRAIN,gamelog_2022.parquet


✅ Required raw NBA columns are present

Available source columns:
['ast', 'ast_pct', 'ast_pct_rank', 'ast_rank', 'ast_ratio', 'ast_ratio_rank', 'ast_to', 'ast_to_rank', 'available_flag', 'blk', 'blk_rank', 'blka', 'blka_rank', 'data_split', 'def_rating', 'def_rating_rank', 'dreb', 'dreb_pct', 'dreb_pct_rank', 'dreb_rank', 'e_def_rating', 'e_net_rating', 'e_off_rating', 'e_pace', 'efg_pct', 'efg_pct_rank', 'fg3_pct', 'fg3_pct_rank', 'fg3a', 'fg3a_rank', 'fg3m', 'fg3m_rank', 'fg_pct', 'fg_pct_rank', 'fga', 'fga_rank', 'fgm', 'fgm_rank', 'ft_pct', 'ft_pct_rank', 'fta', 'fta_rank', 'ftm', 'ftm_rank', 'game_date', 'game_id', 'game_n', 'gp_rank', 'l_rank', 'matchup', 'min', 'min_rank', 'net_rating', 'net_rating_rank', 'off_rating', 'off_rating_rank', 'oreb', 'oreb_pct', 'oreb_pct_rank', 'oreb_rank', 'pace', 'pace_per40', 'pace_rank', 'pf', 'pf_rank', 'pfd', 'pfd_rank', 'pie', 'pie_rank', 'plus_minus', 'plus_minus_rank', 'poss', 'pts', 'pts_rank', 'reb', 'reb_pct', 'reb_pct_rank', 'reb_rank',

,location,rows
0,AWAY,6160
1,HOME,6140


Duplicate team-game rows removed: 0
Rows after de-duplication: 12,300

Critical missing values before removal:


...

⚠️ One or more quality checks did not pass. Review the report before moving to modeling.
✅ Saved clean team-game logs:
/content/againstallodds/data/clean/nba/nba_team_game_logs_clean_2021_22_to_2025_26.csv

✅ Saved clean one-row-per-game dataset:
/content/againstallodds/data/clean/nba/nba_games_clean_2021_22_to_2025_26.csv

✅ Saved team-season summary:
/content/againstallodds/data/clean/nba/nba_team_season_summary_2021_22_to_2025_26.csv

✅ Saved cleaning quality report:
/content/againstallodds/results/week3_nba_cleaning_quality_report.csv
✅ Updated data dictionary saved: /content/againstallodds/docs/data_dictionary_week3.csv


,variable_name,sport,level,category,data_type,description,source,pre_game_only,calculation_or_definition,missing_value_rule,required_or_optional,notes
5,season_start,NBA,game,time,integer,Starting year of the NBA season,Derived from selected season,yes,Example: 2025 represents 2025-26,Must not be missing,required,Used for chronological split
6,season_label,NBA,game,time,string,Readable NBA season label,Derived,yes,YYYY-YY format,Must not be missing,required,Example: 2025-26
7,data_split,NBA,game,validation,string,Chronological TRAIN/VALIDATION/TEST label,Derived,yes,2021-24 train; 2024-25 validation; 2025-26 test,Must not be missing,required,Never randomly shuffle across seasons
8,home_team_id,NBA,game,identifier,integer,NBA team ID for home team,NBA Stats game log,yes,Home row identified from matchup notation,Must not be missing,required,
9,home_team_abbr,NBA,game,identifier,string,Canonical home team abbreviation,NBA Stats game log,yes,Standardized 3-letter abbreviation,Must not be missing,required,
10,home_team_name,NBA,game,identifier,string,Canonical home team name,Derived from abbreviation map,yes,Mapped from team abbreviation,Must not be missing,required,
11,away_team_id,NBA,game,identifier,integer,NBA team ID for away team,NBA Stats game log,yes,Away row identified from matchup notation,Must not be missing,required,
12,away_team_abbr,NBA,game,identifier,string,Canonical away team abbreviation,NBA Stats game log,yes,Standardized 3-letter abbreviation,Must not be missing,required,
13,away_team_name,NBA,game,identifier,string,Canonical away team name,Derived from abbreviation map,yes,Mapped from team abbreviation,Must not be missing,required,
14,home_pts,NBA,game,outcome,numeric,Final points scored by home team,NBA Stats game log,no,Final game score,Must not be missing,required,Post-game value; not a same-game predictor


✅ Documentation saved: /content/againstallodds/docs/week3_nba_source_and_cleaning_plan.md


,deliverable,created
0,clean_team_game_logs,True
1,clean_game_dataset,True
2,team_season_summary,True
3,updated_data_dictionary,True
4,missingness_report,True
5,coverage_report,True
6,quality_report,True
7,source_documentation,True


AGAINST ALL ODDS — WEEK 3 NBA DATA CLEANING
⚠️ WEEK 3 COMPLETED WITH ITEMS TO REVIEW

Clean games: 6,140
Clean team-game rows: 12,280
Seasons: 5
Date range: 2021-10-19 to 2026-04-12

Next: run 04_nba_analysis.ipynb


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import sys
import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)

PROJECT_TITLE = "Against All Odds"
RANDOM_SEED = 42

def running_in_colab():
    return "google.colab" in sys.modules

CURRENT_DIR = Path.cwd().resolve()

KNOWN_REPO_NAMES = {
    "against-all-odds",
    "againstallodds",
    "Against-All-Odds",
    "Against_All_Odds",
    "Against All Odds",
}

if CURRENT_DIR.name in KNOWN_REPO_NAMES:
    PROJECT_ROOT = CURRENT_DIR
elif running_in_colab():
    PROJECT_ROOT = Path("/content/againstallodds")
else:
    PROJECT_ROOT = CURRENT_DIR / "againstallodds"

CLEAN_DIR = PROJECT_ROOT / "data" / "clean" / "nba"
DOCS_DIR = PROJECT_ROOT / "docs"
RESULTS_DIR = PROJECT_ROOT / "results"

GAME_PATH = CLEAN_DIR / "nba_games_clean_2021_22_to_2025_26.csv"
TEAM_LOG_PATH = CLEAN_DIR / "nba_team_game_logs_clean_2021_22_to_2025_26.csv"
TEAM_SUMMARY_PATH = CLEAN_DIR / "nba_team_season_summary_2021_22_to_2025_26.csv"

for required_path in [GAME_PATH, TEAM_LOG_PATH, TEAM_SUMMARY_PATH]:
    if not required_path.exists():
        raise FileNotFoundError(
            f"Missing {required_path}. Run 03_data_cleaning.ipynb first."
        )

print(f"✅ Project root: {PROJECT_ROOT}")

# ============================================================
# 2. Load cleaned Week 3 datasets
# ============================================================

games = pd.read_csv(
    GAME_PATH,
    parse_dates=["game_date"]
)

team_logs = pd.read_csv(
    TEAM_LOG_PATH,
    parse_dates=["game_date"]
)

team_summary = pd.read_csv(
    TEAM_SUMMARY_PATH
)

print("Clean game dataset:", games.shape)
print("Clean team-game dataset:", team_logs.shape)
print("Team-season summary:", team_summary.shape)

display(games.head())

# ============================================================
# 3. Dataset overview
# ============================================================

overview = pd.DataFrame([
    {
        "dataset": "nba_games_clean",
        "rows": len(games),
        "columns": len(games.columns),
        "first_date": games["game_date"].min(),
        "last_date": games["game_date"].max(),
        "unique_games": games["game_id"].nunique(),
        "seasons": games["season_label"].nunique(),
    },
    {
        "dataset": "nba_team_game_logs_clean",
        "rows": len(team_logs),
        "columns": len(team_logs.columns),
        "first_date": team_logs["game_date"].min(),
        "last_date": team_logs["game_date"].max(),
        "unique_games": team_logs["game_id"].nunique(),
        "seasons": team_logs["season_label"].nunique(),
    },
])

display(overview)

# ============================================================
# 4. Season-by-season quality report
# ============================================================

season_rows = []

for season_label, part in games.groupby("season_label", sort=True):
    teams = set(part["home_team_abbr"]).union(set(part["away_team_abbr"]))

    required_cols = [
        "game_id",
        "game_date",
        "home_team_abbr",
        "away_team_abbr",
        "home_pts",
        "away_pts",
        "HOME_WIN",
    ]

    season_rows.append({
        "season_label": season_label,
        "games": part["game_id"].nunique(),
        "expected_games": 1230,
        "game_count_pass": part["game_id"].nunique() == 1230,
        "teams": len(teams),
        "team_count_pass": len(teams) == 30,
        "duplicate_game_ids": int(part["game_id"].duplicated().sum()),
        "required_missing_cells": int(part[required_cols].isna().sum().sum()),
        "first_game": part["game_date"].min(),
        "last_game": part["game_date"].max(),
    })

season_quality = pd.DataFrame(season_rows)

display(season_quality)

season_quality_path = RESULTS_DIR / "week3_nba_season_quality.csv"
season_quality.to_csv(season_quality_path, index=False)

print(f"✅ Saved: {season_quality_path}")

# ============================================================
# 5. Validate HOME_WIN and home/away structure
# ============================================================

validation_checks = {
    "game_id_unique": games["game_id"].is_unique,
    "date_sorted": games["game_date"].is_monotonic_increasing,
    "home_win_binary": set(games["HOME_WIN"].dropna().unique()).issubset({0, 1}),
    "home_flag_all_one": games["home_is_home"].eq(1).all(),
    "away_flag_all_zero": games["away_is_home"].eq(0).all(),
    "home_and_away_different": (
        games["home_team_id"] != games["away_team_id"]
    ).all(),
    "home_win_matches_scores": (
        games["HOME_WIN"]
        == (games["home_pts"] > games["away_pts"]).astype(int)
    ).all(),
    "no_tied_games": (games["home_pts"] != games["away_pts"]).all(),
    "five_seasons": games["season_label"].nunique() == 5,
}

validation_df = pd.DataFrame([
    {"check": key, "passed": bool(value)}
    for key, value in validation_checks.items()
])

display(validation_df)

# ============================================================
# 6. Team coverage by season
# ============================================================

team_coverage_rows = []

for season_label, part in team_logs.groupby("season_label", sort=True):
    team_counts = (
        part.groupby("team_abbreviation")["game_id"]
        .nunique()
        .sort_values()
    )

    for team_abbr, games_played in team_counts.items():
        team_coverage_rows.append({
            "season_label": season_label,
            "team_abbreviation": team_abbr,
            "games_played": int(games_played),
            "expected_games": 82,
            "82_game_schedule_pass": int(games_played) == 82,
        })

team_coverage = pd.DataFrame(team_coverage_rows)

coverage_summary = (
    team_coverage.groupby("season_label", as_index=False)
    .agg(
        teams=("team_abbreviation", "nunique"),
        min_games=("games_played", "min"),
        max_games=("games_played", "max"),
        teams_with_82=("82_game_schedule_pass", "sum"),
    )
)

display(coverage_summary)

if not team_coverage["82_game_schedule_pass"].all():
    print(
        "⚠️ Some teams do not show exactly 82 regular-season games. "
        "Inspect before modeling."
    )
else:
    print("✅ Every team has 82 regular-season games in every selected season")

# ============================================================
# 7. Basic season summaries for quality control
# ============================================================

season_summary = (
    games.groupby(
        ["season_start", "season_label", "data_split"],
        as_index=False
    )
    .agg(
        games=("game_id", "nunique"),
        home_win_rate=("HOME_WIN", "mean"),
        avg_home_points=("home_pts", "mean"),
        avg_away_points=("away_pts", "mean"),
        avg_home_margin=("HOME_MARGIN", "mean"),
        avg_total_points=("GAME_TOTAL_POINTS", "mean"),
    )
)

for col in [
    "home_win_rate",
    "avg_home_points",
    "avg_away_points",
    "avg_home_margin",
    "avg_total_points",
]:
    season_summary[col] = season_summary[col].round(3)

display(season_summary)

season_summary_path = RESULTS_DIR / "week3_nba_basic_season_summary.csv"
season_summary.to_csv(season_summary_path, index=False)

print(f"✅ Saved: {season_summary_path}")

# ============================================================
# 8. Team-season standings-style validation
# ============================================================

standings_view = (
    team_summary[
        [
            "season_label",
            "team_abbreviation",
            "team_name",
            "games",
            "wins",
            "losses",
            "win_pct",
            "avg_points",
        ]
        + (
            ["avg_points_allowed"]
            if "avg_points_allowed" in team_summary.columns
            else []
        )
        + (
            ["avg_point_differential"]
            if "avg_point_differential" in team_summary.columns
            else []
        )
    ]
    .sort_values(
        ["season_label", "win_pct"],
        ascending=[True, False]
    )
    .reset_index(drop=True)
)

display(standings_view.head(30))

# ============================================================
# 9. Check availability of future Week 6 feature inputs
# ============================================================

future_stat_candidates = [
    "pts",
    "opp_pts",
    "plus_minus",
    "fg_pct",
    "fg3_pct",
    "ft_pct",
    "reb",
    "ast",
    "stl",
    "blk",
    "tov",
    "off_rating",
    "def_rating",
    "net_rating",
    "efg_pct",
    "ts_pct",
    "pace",
    "poss",
]

availability_rows = []

for col in future_stat_candidates:
    availability_rows.append({
        "field": col,
        "available": col in team_logs.columns,
        "non_missing_pct": (
            round(
                float(team_logs[col].notna().mean() * 100),
                2
            )
            if col in team_logs.columns
            else 0.0
        ),
        "same_game_predictor_allowed": "NO",
        "week6_rule": (
            "Use historical values only; shift(1) before rolling."
        ),
    })

stat_availability = pd.DataFrame(availability_rows)

display(stat_availability)

availability_path = DOCS_DIR / "week3_nba_stat_availability.csv"
stat_availability.to_csv(availability_path, index=False)

print(f"✅ Saved: {availability_path}")

# ============================================================
# 10. Map cleaned data to planned capstone features
# ============================================================

feature_readiness = pd.DataFrame([
    {
        "planned_feature": "Win percentage",
        "source_in_clean_data": "wl / team_win",
        "status": "READY_FOR_WEEK6",
        "week6_method": "Shift team_win by 1 game, then expanding/rolling mean",
    },
    {
        "planned_feature": "Point differential",
        "source_in_clean_data": "plus_minus or pts - opp_pts",
        "status": "READY_FOR_WEEK6",
        "week6_method": "Shift by 1 game before rolling/expanding mean",
    },
    {
        "planned_feature": "Elo rating",
        "source_in_clean_data": "game outcomes + chronological order",
        "status": "READY_FOR_WEEK8",
        "week6_method": "Maintain rating before updating on current result",
    },
    {
        "planned_feature": "Recent form — last 3",
        "source_in_clean_data": "team_win",
        "status": "READY_FOR_WEEK6",
        "week6_method": "team_win.shift(1).rolling(3)",
    },
    {
        "planned_feature": "Recent form — last 5",
        "source_in_clean_data": "team_win",
        "status": "READY_FOR_WEEK6",
        "week6_method": "team_win.shift(1).rolling(5)",
    },
    {
        "planned_feature": "Recent form — last 10",
        "source_in_clean_data": "team_win",
        "status": "READY_FOR_WEEK6",
        "week6_method": "team_win.shift(1).rolling(10)",
    },
    {
        "planned_feature": "Recent scoring",
        "source_in_clean_data": "pts",
        "status": "READY_FOR_WEEK6",
        "week6_method": "pts.shift(1).rolling(window)",
    },
    {
        "planned_feature": "Recent defensive performance",
        "source_in_clean_data": "opp_pts / def_rating",
        "status": "READY_FOR_WEEK6",
        "week6_method": "Shift by 1 game before rolling",
    },
    {
        "planned_feature": "Offensive rating",
        "source_in_clean_data": "off_rating if available",
        "status": "SOURCE_CHECKED",
        "week6_method": "Historical shifted rolling mean",
    },
    {
        "planned_feature": "Effective FG%",
        "source_in_clean_data": "efg_pct if available",
        "status": "SOURCE_CHECKED",
        "week6_method": "Historical shifted rolling mean",
    },
    {
        "planned_feature": "Three-point percentage",
        "source_in_clean_data": "fg3_pct",
        "status": "READY_FOR_WEEK6",
        "week6_method": "Historical shifted rolling mean",
    },
    {
        "planned_feature": "Home advantage",
        "source_in_clean_data": "is_home / home-away game structure",
        "status": "READY",
        "week6_method": "Game-level context known before tipoff",
    },
    {
        "planned_feature": "Days of rest",
        "source_in_clean_data": "game_date",
        "status": "READY_FOR_WEEK6",
        "week6_method": "Difference from team's prior game date",
    },
    {
        "planned_feature": "Opponent quality",
        "source_in_clean_data": "opponent historical record/Elo",
        "status": "READY_FOR_WEEK6",
        "week6_method": "Use opponent pre-game strength only",
    },
])

display(feature_readiness)

feature_readiness_path = DOCS_DIR / "week3_nba_feature_readiness.csv"
feature_readiness.to_csv(feature_readiness_path, index=False)

print(f"✅ Saved: {feature_readiness_path}")

# ============================================================
# 11. Save concise Week 3 NBA dataset summary
# ============================================================

def dataframe_to_markdown(df):
    try:
        return df.to_markdown(index=False)
    except Exception:
        return "```text\n" + df.to_string(index=False) + "\n```"


summary_md = f'''# Against All Odds — Week 3 NBA Dataset Summary

Generated: {datetime.now(timezone.utc).isoformat(timespec="seconds")}

## Clean datasets

- Game-level rows: {len(games):,}
- Team-game rows: {len(team_logs):,}
- Seasons: {games["season_label"].nunique()}
- First game date: {games["game_date"].min().date()}
- Last game date: {games["game_date"].max().date()}

## Selected seasons

{dataframe_to_markdown(season_quality)}

## Core validation

{dataframe_to_markdown(validation_df)}

## Leakage reminder

Current-game scores and box-score statistics are outcomes/post-game information.
They are stored so they can become historical inputs for later games.
Week 6 must apply shift(1) before rolling calculations.

## Week 3 conclusion

The cleaned NBA historical dataset is ready for the later feature-engineering
stage if all validation checks pass.
'''

summary_path = DOCS_DIR / "week3_nba_dataset_summary.md"
summary_path.write_text(summary_md, encoding="utf-8")

print(f"✅ Saved: {summary_path}")

# ============================================================
# 12. Final Week 3 deliverable status
# ============================================================

final_checks = {
    "clean_game_dataset_exists": GAME_PATH.exists(),
    "clean_team_log_exists": TEAM_LOG_PATH.exists(),
    "team_summary_exists": TEAM_SUMMARY_PATH.exists(),
    "five_seasons_present": games["season_label"].nunique() == 5,
    "30_teams_each_season": season_quality["team_count_pass"].all(),
    "1230_games_each_season": season_quality["game_count_pass"].all(),
    "core_validation_passes": validation_df["passed"].all(),
    "feature_readiness_saved": feature_readiness_path.exists(),
    "dataset_summary_saved": summary_path.exists(),
}

final_status = pd.DataFrame([
    {"deliverable_check": key, "passed": bool(value)}
    for key, value in final_checks.items()
])

display(final_status)

print("=" * 78)
print("AGAINST ALL ODDS — WEEK 3 FINAL STATUS")
print("=" * 78)

if final_status["passed"].all():
    print("✅ WEEK 3 DELIVERABLE COMPLETE: CLEAN HISTORICAL NBA DATASET")
else:
    print("⚠️ REVIEW FAILED CHECKS BEFORE MOVING FORWARD")

print("\nWeek 4 according to the capstone schedule: NFL collection and cleaning.")

✅ Project root: /content/againstallodds
Clean game dataset: (6140, 72)
Clean team-game dataset: (12280, 102)
Team-season summary: (150, 27)


,game_id,season_start,season_end,season_label,data_split,game_date,home_team_id,home_team_abbr,home_team_name,home_wl,...,away_efg_pct,away_ts_pct,away_pace,away_poss,home_is_home,away_is_home,HOME_WIN,HOME_MARGIN,GAME_TOTAL_POINTS,score_quality
0,22100001,2021,2022,2021-22,TRAIN,2021-10-19,1610612749,MIL,Milwaukee Bucks,W,...,0.542,0.552,102.00,102,1,0,1,23,231,OK
1,22100002,2021,2022,2021-22,TRAIN,2021-10-19,1610612747,LAL,Los Angeles Lakers,L,...,0.516,0.570,112.50,113,1,0,0,-7,235,OK
2,22100003,2021,2022,2021-22,TRAIN,2021-10-20,1610612766,CHA,Charlotte Hornets,W,...,0.561,0.607,106.50,107,1,0,1,1,245,OK
3,22100004,2021,2022,2021-22,TRAIN,2021-10-20,1610612765,DET,Detroit Pistons,L,...,0.471,0.508,98.00,98,1,0,0,-6,182,OK
4,22100005,2021,2022,2021-22,TRAIN,2021-10-20,1610612752,NYK,New York Knicks,W,...,0.500,0.527,103.03,124,1,0,1,4,272,OK


,dataset,rows,columns,first_date,last_date,unique_games,seasons
0,nba_games_clean,6140,72,2021-10-19,2026-04-12,6140,5
1,nba_team_game_logs_clean,12280,102,2021-10-19,2026-04-12,6140,5


,season_label,games,expected_games,game_count_pass,teams,team_count_pass,duplicate_game_ids,required_missing_cells,first_game,last_game
0,2021-22,1230,1230,True,30,True,0,0,2021-10-19,2022-04-10
1,2022-23,1230,1230,True,30,True,0,0,2022-10-18,2023-04-09
2,2023-24,1230,1230,True,30,True,0,0,2023-10-24,2024-04-14
3,2024-25,1225,1230,False,30,True,0,0,2024-10-22,2025-04-13
4,2025-26,1225,1230,False,30,True,0,0,2025-10-21,2026-04-12


✅ Saved: /content/againstallodds/results/week3_nba_season_quality.csv


...

✅ Saved: /content/againstallodds/docs/week3_nba_stat_availability.csv


,planned_feature,source_in_clean_data,status,week6_method
0,Win percentage,wl / team_win,READY_FOR_WEEK6,"Shift team_win by 1 game, then expanding/rolli..."
1,Point differential,plus_minus or pts - opp_pts,READY_FOR_WEEK6,Shift by 1 game before rolling/expanding mean
2,Elo rating,game outcomes + chronological order,READY_FOR_WEEK8,Maintain rating before updating on current result
3,Recent form — last 3,team_win,READY_FOR_WEEK6,team_win.shift(1).rolling(3)
4,Recent form — last 5,team_win,READY_FOR_WEEK6,team_win.shift(1).rolling(5)
5,Recent form — last 10,team_win,READY_FOR_WEEK6,team_win.shift(1).rolling(10)
6,Recent scoring,pts,READY_FOR_WEEK6,pts.shift(1).rolling(window)
7,Recent defensive performance,opp_pts / def_rating,READY_FOR_WEEK6,Shift by 1 game before rolling
8,Offensive rating,off_rating if available,SOURCE_CHECKED,Historical shifted rolling mean
9,Effective FG%,efg_pct if available,SOURCE_CHECKED,Historical shifted rolling mean


✅ Saved: /content/againstallodds/docs/week3_nba_feature_readiness.csv
✅ Saved: /content/againstallodds/docs/week3_nba_dataset_summary.md


,deliverable_check,passed
0,clean_game_dataset_exists,True
1,clean_team_log_exists,True
2,team_summary_exists,True
3,five_seasons_present,True
4,30_teams_each_season,True
5,1230_games_each_season,False
6,core_validation_passes,True
7,feature_readiness_saved,True
8,dataset_summary_saved,True


AGAINST ALL ODDS — WEEK 3 FINAL STATUS
⚠️ REVIEW FAILED CHECKS BEFORE MOVING FORWARD

Week 4 according to the capstone schedule: NFL collection and cleaning.
